In [1]:
import pandas as pd

# 1. Đọc dữ liệu từ file promotions.csv
df_promo = pd.read_csv('promotions.csv')
print(f"Tổng số chương trình khuyến mãi ban đầu: {len(df_promo)}")

# 2. Đổi tên cột cho khớp với Schema yêu cầu
df_promo.rename(columns={
    'promo_id': 'Promotion_Code',  # Mã text như PROMO-0001
    'discount_value': 'Discount_Rate',
    'start_date': 'Start_Date',
    'end_date': 'End_Date'
}, inplace=True)

# 3. Xử lý Missing Data (Dữ liệu rỗng)
# Các giá trị Null trong 'applicable_category' mang ý nghĩa áp dụng cho mọi danh mục
df_promo['applicable_category'] = df_promo['applicable_category'].fillna('All Categories')

# 4. Tạo Khóa chính (Primary Key): Promotion_ID (Số nguyên tự tăng)
df_promo.insert(0, 'Promotion_ID', range(1, len(df_promo) + 1))

# 5. Sắp xếp đúng thứ tự cấu trúc (Schema)
final_promo_columns = [
    'Promotion_ID', 'Promotion_Code', 'Discount_Rate', 'Start_Date', 
    'End_Date', 'applicable_category', 'promo_channel', 
    'stackable_flag', 'min_order_value'
]
df_promo_final = df_promo[final_promo_columns]

print("\nDữ liệu mẫu bảng PROMOTIONS sau khi làm sạch:")
display(df_promo_final.head())

Tổng số chương trình khuyến mãi ban đầu: 50

Dữ liệu mẫu bảng PROMOTIONS sau khi làm sạch:


,Promotion_ID,Promotion_Code,Discount_Rate,Start_Date,End_Date,applicable_category,promo_channel,stackable_flag,min_order_value
0,1,PROMO-0001,12.0,2013-03-18,2013-04-17,All Categories,email,1,0
1,2,PROMO-0002,18.0,2013-06-23,2013-07-22,All Categories,online,0,0
2,3,PROMO-0003,10.0,2013-08-30,2013-10-02,All Categories,email,0,0
3,4,PROMO-0004,20.0,2013-11-18,2014-01-02,All Categories,all_channels,0,50000
4,5,PROMO-0005,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000


In [2]:
# 1. Đọc dữ liệu order_items (đặt low_memory=False vì file có thể lớn)
df_order_items = pd.read_csv('order_items.csv', low_memory=False)

# 2. Tạo từ điển ánh xạ (Mapping dictionary) từ Promotion_Code -> Promotion_ID
promo_mapping = dict(zip(df_promo_final['Promotion_Code'], df_promo_final['Promotion_ID']))

# 3. Ánh xạ Khóa ngoại (Foreign Keys) sang bảng order_items
df_order_items['Promotion_ID'] = df_order_items['promo_id'].map(promo_mapping)
df_order_items['Promotion_ID_2'] = df_order_items['promo_id_2'].map(promo_mapping)

# 4. Ép kiểu dữ liệu sang Int64 (Integer hỗ trợ giá trị Null của Pandas)
# Do không phải đơn hàng nào cũng dùng mã khuyến mãi (sẽ có giá trị NaN)
df_order_items['Promotion_ID'] = df_order_items['Promotion_ID'].astype('Int64')
df_order_items['Promotion_ID_2'] = df_order_items['Promotion_ID_2'].astype('Int64')

print("\nDữ liệu mẫu bảng ORDER_ITEMS sau khi ánh xạ Khóa ngoại 1-N:")
# Lọc ra các dòng có dùng mã để xem kết quả trực quan
display(df_order_items[['order_id', 'promo_id', 'Promotion_ID']].dropna().head())



Dữ liệu mẫu bảng ORDER_ITEMS sau khi ánh xạ Khóa ngoại 1-N:


,order_id,promo_id,Promotion_ID
41316,46253,PROMO-0006,6
41317,46254,PROMO-0006,6
41319,46257,PROMO-0006,6
41320,46257,PROMO-0006,6
41321,46258,PROMO-0006,6


In [ ]:
# 1. Đảm bảo Promotion_ID và Promotion_Code trong bảng PROMOTIONS là duy nhất tuyệt đối
assert df_promo_final['Promotion_ID'].is_unique, "Lỗi: Promotion_ID bị trùng lặp!"
assert df_promo_final['Promotion_Code'].is_unique, "Lỗi: Promotion_Code bị trùng lặp!"
print("\n👉🏿 Ràng buộc Unique (1-N) cho bảng Khuyến mãi đã được xác thực.")

# 2. Xuất dữ liệu ra file chuẩn hóa
df_promo_final.to_csv('promotions_processed.csv', index=False, encoding='utf-8')
df_order_items.to_csv('order_items_processed.csv', index=False, encoding='utf-8')

print("👉🏿 Đã lưu toàn bộ dữ liệu sạch vào file CSV!")


👉🏿 Ràng buộc Unique (1-N) cho bảng Khuyến mãi đã được xác thực.
